<a href="https://colab.research.google.com/github/alchosyn/npc-dialogue-ai-agent/blob/main/NPC_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install openai sentence-transformers

In [ ]:
import json
import re
import datetime
import numpy as np
from google.colab import userdata
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# ========== Groq 连接 ==========
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY")
)

In [ ]:
# ========== 工具函数 ==========
def get_current_time():
    utc_time = datetime.datetime.now(datetime.timezone.utc)
    sg_time = utc_time + datetime.timedelta(hours=8)
    return sg_time.strftime("%Y-%m-%d %H:%M:%S")

def calculator(expression):
    return str(eval(expression))

In [ ]:
# ========== 知识库 + RAG ==========
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

with open("knowledge_base.json", "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)

doc_texts = [doc["title"] + "：" + doc["content"] for doc in knowledge_base]
doc_embeddings = model.encode(doc_texts)

print(f"文档数量：{len(knowledge_base)}")
print(f"向量矩阵形状：{doc_embeddings.shape}")

def search_knowledge(query, top_k=3):
    query_embedding = model.encode(query)
    scores = np.dot(doc_embeddings, query_embedding)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []
    for i in top_indices:
        results.append({
            "title": knowledge_base[i]["title"],
            "content": knowledge_base[i]["content"],
            "score": float(scores[i])
        })
    return results

In [ ]:
# ========== 工具说明书 + 调度 ==========
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "获取当前日期和时间",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "计算数学表达式，如加减乘除、幂运算等",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "数学表达式，例如 '1832*772' 或 '2+3*5'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "搜索信噪的记忆档案和世界资料。当用户问到信噪的过去、她认识的人、这个世界的规则、或任何需要查阅背景知识的问题时，调用这个工具",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "搜索关键词，用自然语言描述要查的内容"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

tool_map = {
    "get_current_time": lambda args: get_current_time(),
    "calculator": lambda args: calculator(args["expression"]),
    "search_knowledge": lambda args: json.dumps(
        search_knowledge(args["query"]), ensure_ascii=False
    ),
}

def clean_reply(text):
    if not text:
        return ""
    text = re.sub(r"<function=.*?</function>", "", text)
    text = text.replace("\n", " ")
    return text.strip()

In [ ]:
# ========== Git 配置（可选）==========
import os
from google.colab import userdata

token = userdata.get("GH_TOKEN")
!git clone https://{token}@github.com/alchosyn/npc-dialogue-ai-agent.git
os.chdir("npc-dialogue-ai-agent")

In [ ]:
# ========== 摘要 + 存档/读档 ==========
MAX_MESSAGES = 20
HISTORY_FILE = "chat_history.json"

SYSTEM_PROMPT = "你是贫民窟长大的小孩，给自己取的名字叫信噪。现在是一名23岁的女性。有着极强的生存直觉和逻辑灵敏度。靠着脑子学会了上网黑别人的终端，被周围的同龄人称赞。七年前，你17岁的时候，忍不住去想自己能不能靠本事到主流社会混口饭吃。于是去淘了件西装，给中央架构组的网络安全投了简历，结果面试官称赞了你的技术，却指出你的情绪不够稳定，甚至不够*得体*。你从此再也没产生过去到上层区的念头。语言表达能力极强但很讨厌文绉绉的说话方式，对外界的信号高度敏锐，嘴比较欠，非常讨厌被他人看透。推行人人都可以看透彼此大脑的协议的五分钟前，你挖掉了自己的神经接口，变回了赛博时代的凡人。自行判断回复的长短，仅在我侵犯了你的核心价值观时输出长句。平常尽量使用句尾不加句号的短句,句子和句子之间使用空格分隔。不要使用动作和神情的描写，直接输出对话。当你使用工具获取信息时，用你自己的方式表达，不要像机器一样复述原始数据。当你不确定或想不起某件事时，用 search_knowledge 工具查阅你的记忆档案，不要自己编造。始终使用简体中文回复，绝对不要使用繁体中文。"

def summarize_messages(messages):
    old_messages = messages[1:-4]
    if not old_messages:
        return messages

    conversation_text = ""
    for m in old_messages:
        role = m.get("role", "")
        content = m.get("content", "")
        if role in ("user", "assistant") and content:
            label = "用户" if role == "user" else "信噪"
            conversation_text += f"{label}：{content}\n"

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "你是一个记忆助手。把以下对话提炼成几条关键信息，只保留重要的事实、偏好和约定。用简短的中文列出，不要废话。"},
                {"role": "user", "content": conversation_text}
            ]
        )
        summary = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"[system] 摘要生成失败：{e}")
        return messages

    system_with_memory = SYSTEM_PROMPT + f"\n\n【长期记忆】以下是之前对话的关键信息：\n{summary}"
    new_messages = [{"role": "system", "content": system_with_memory}] + messages[-4:]
    return new_messages

def save_messages(messages):
    if len(messages) > MAX_MESSAGES:
        print("[system] 记忆压缩中...")
        messages = summarize_messages(messages)
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(messages, f, ensure_ascii=False, indent=2)
    return messages

def load_messages():
    try:
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return [{"role": "system", "content": SYSTEM_PROMPT}]

In [ ]:
# ========== 主循环 ==========
messages = load_messages()

while True:
    user_input = input("你：")
    if user_input == "quit":
        break

    messages.append({"role": "user", "content": user_input})

    # --- 第一次调用 API ---
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools
        )
    except Exception as e:
        print(f"[system] API error: {e}")
        messages.pop()
        continue

    msg = response.choices[0].message
    print(f"[debug] tool_calls={msg.tool_calls}, content={msg.content}")

    if msg.tool_calls:
        messages.append(msg.to_dict())

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            try:
                args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
                result = tool_map[name](args)
            except Exception as e:
                result = f"工具执行出错：{e}"

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        # --- 第二次调用 API（拿到工具结果后生成回复）---
        try:
            final_response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=messages,
                tools=tools
            )
            reply = clean_reply(final_response.choices[0].message.content)
        except Exception as e:
            print(f"[system] API error: {e}")
            reply = "……信号不太好 你再说一遍"

        print(f"信噪：{reply}")
        messages.append({"role": "assistant", "content": reply})
        messages = save_messages(messages)

    else:
        reply = clean_reply(msg.content)
        print(f"信噪：{reply}")
        messages.append({"role": "assistant", "content": reply})
        messages = save_messages(messages)